# 08 — Modeling

**Tujuan notebook ini (Fase 7.5-7.7 roadmap):**
- Load `ml_dataset_observation.csv` (hasil Decision Gate, notebook 07)
- Preprocessing (handle missing value tanpa `fillna(0)` sembarangan)
- Fit 3 model: **Logistic Regression** (baseline), **Random Forest**, **XGBoost**
- Semua model pakai class imbalance handling (`class_weight='balanced'` / `scale_pos_weight`) — imbalance 81:1 sudah dikonfirmasi di Decision Gate, BUKAN alasan untuk mundur dari classification.
- Model & prediksi disimpan untuk evaluasi mendalam di `09_model_evaluation.ipynb`

> Evaluasi metrik lengkap (ROC-AUC, PR-AUC, calibration) dilakukan di notebook 09, BUKAN di sini — notebook ini fokus pada fitting model dengan benar.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data_processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

ml_dataset = pd.read_csv(DATA_DIR / "ml_dataset_observation.csv", parse_dates=["last_order_date_observation"])
print("ml_dataset shape:", ml_dataset.shape)
print(f"Positive rate: {ml_dataset['target'].mean()*100:.2f}%")
ml_dataset.head()


ml_dataset shape: (54738, 10)
Positive rate: 1.21%


,customer_unique_id,order_count,total_spending,avg_order_value,last_order_date_observation,avg_review_score,avg_delivery_days,unique_categories,days_since_last_order,target
0,0000f46a3911fa3c0805444483337064,1,86.22,86.22,2017-03-10 21:05:03,3.0,25.0,1.0,355,0
1,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,2017-10-12 20:29:41,4.0,20.0,1.0,139,0
2,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,2017-11-14 19:45:42,5.0,13.0,1.0,106,0
3,00053a61a98854899e70ed204dd4bafe,1,419.18,419.18,2018-02-28 11:15:41,1.0,16.0,1.0,0,0
4,0005e1862207bf6ccc02e4228effd9a0,1,150.12,150.12,2017-03-04 23:32:12,4.0,4.0,1.0,361,0


---
## 1. Cek Missing Value Sebelum Modeling

Fitur seperti `avg_delivery_days` bisa NaN (customer yang order-nya belum/tidak delivered saat observation cutoff) — **bukan salah input**, tapi representasi kondisi valid. Ditangani lewat imputasi eksplisit, bukan `fillna(0)` yang menyesatkan makna.


In [2]:
feature_cols = [
    "order_count", "total_spending", "avg_order_value", "avg_review_score",
    "avg_delivery_days", "unique_categories", "days_since_last_order",
]

print("Missing value per feature:")
print(ml_dataset[feature_cols].isnull().sum())
print(f"\nTotal baris: {len(ml_dataset)}")


Missing value per feature:
order_count                 0
total_spending              0
avg_order_value             0
avg_review_score          458
avg_delivery_days        1759
unique_categories           0
days_since_last_order       0
dtype: int64

Total baris: 54738


---
## 2. Train / Validation / Test Split

Split 3 arah (bukan cuma train/test), sesuai roadmap Fase 7.4. Stratified untuk MVP awal (mempertahankan proporsi positive class di tiap split) — catatan: rolling/expanding time split bisa jadi enhancement lanjutan kalau diperlukan.


In [3]:
X = ml_dataset[feature_cols]
y = ml_dataset["target"]

# Split 1: train+val (80%) vs test (20%) - test benar-benar untouched sampai evaluasi akhir
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Split 2: train (80% dari trainval) vs validation (20% dari trainval) - untuk model selection
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=42, stratify=y_trainval
)

print(f"Train      : {len(X_train):,} ({y_train.mean()*100:.2f}% positive)")
print(f"Validation : {len(X_val):,} ({y_val.mean()*100:.2f}% positive)")
print(f"Test       : {len(X_test):,} ({y_test.mean()*100:.2f}% positive) -- UNTOUCHED sampai notebook 09")


Train      : 35,032 (1.21% positive)
Validation : 8,758 (1.21% positive)
Test       : 10,948 (1.21% positive) -- UNTOUCHED sampai notebook 09


---
## 3. Preprocessing Pipeline

Imputasi median untuk missing value (robust terhadap outlier), scaling untuk Logistic Regression (RF & XGBoost tidak butuh scaling tapi tidak masalah kalau tetap diberi data yang sudah diimpute).


In [4]:
preprocessor = ColumnTransformer(transformers=[
    ("impute_scale", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_cols)
])

# Untuk tree-based model (RF, XGBoost): imputasi saja, tanpa scaling (tidak diperlukan)
preprocessor_tree = ColumnTransformer(transformers=[
    ("impute", SimpleImputer(strategy="median"), feature_cols)
])

print("Preprocessor siap.")


Preprocessor siap.


---
## 4. Model 1 — Logistic Regression (Baseline)

`class_weight="balanced"` untuk menangani imbalance 81:1 tanpa mengubah data (bukan SMOTE).


In [5]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

logreg_pipeline.fit(X_train, y_train)

val_proba_logreg = logreg_pipeline.predict_proba(X_val)[:, 1]
print("Logistic Regression fitted.")
print(f"Distribusi predicted probability di validation set: "
      f"min={val_proba_logreg.min():.4f}, max={val_proba_logreg.max():.4f}, mean={val_proba_logreg.mean():.4f}")


Logistic Regression fitted.
Distribusi predicted probability di validation set: min=0.0183, max=0.9985, mean=0.4818


---
## 5. Model 2 — Random Forest

`class_weight="balanced"` juga, untuk menangkap nonlinearity & interaction antar fitur.


In [6]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("classifier", RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced",
        random_state=42, n_jobs=-1
    )),
])

rf_pipeline.fit(X_train, y_train)

val_proba_rf = rf_pipeline.predict_proba(X_val)[:, 1]
print("Random Forest fitted.")
print(f"Distribusi predicted probability di validation set: "
      f"min={val_proba_rf.min():.4f}, max={val_proba_rf.max():.4f}, mean={val_proba_rf.mean():.4f}")


Random Forest fitted.
Distribusi predicted probability di validation set: min=0.0451, max=0.8661, mean=0.4117


---
## 6. Model 3 — XGBoost

`scale_pos_weight` = rasio negative:positive di training set, untuk menangani imbalance tanpa SMOTE.


In [7]:
import sys
try:
    import xgboost as xgb
except ModuleNotFoundError:
    !{sys.executable} -m pip install xgboost
    import xgboost as xgb

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight yang dipakai: {scale_pos_weight:.2f}")

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("classifier", xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",  # PR-AUC sebagai metrik internal, konsisten dengan Fase 8
        random_state=42, n_jobs=-1,
    )),
])

xgb_pipeline.fit(X_train, y_train)

val_proba_xgb = xgb_pipeline.predict_proba(X_val)[:, 1]
print("XGBoost fitted.")
print(f"Distribusi predicted probability di validation set: "
      f"min={val_proba_xgb.min():.4f}, max={val_proba_xgb.max():.4f}, mean={val_proba_xgb.mean():.4f}")


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 750.4 kB/s eta 0:01:05
   ---------------------------------------- 0.5/48.9 MB 750.4 kB/s eta 0:01:05
   ---------------------------------------- 0.5/48.9 MB 750.4 kB/s eta 0:01:05
   ---------------------------------------- 0.5/48.9 MB 750.4 kB/s eta 0:01:05
   ---------------------------------------- 0.5/48.9 MB 750.4 kB/s eta 0:01:05
    --------------------------------------- 0.8/48.9 MB 351.1 kB/s eta 0:02:18
    --------------------------------------- 0.8/48.9 MB 351.1 kB/s eta 0:02:18
    ---------------

---
## 7. Sanity Check Cepat (Preview) — Evaluasi Lengkap di Notebook 09

Ini cuma preview cepat pakai `roc_auc_score` biasa, BUKAN evaluasi final. PR-AUC, calibration, dan threshold tuning yang sebenarnya dilakukan di `09_model_evaluation.ipynb`.


In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score

preview = pd.DataFrame({
    "model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "roc_auc_preview": [
        roc_auc_score(y_val, val_proba_logreg),
        roc_auc_score(y_val, val_proba_rf),
        roc_auc_score(y_val, val_proba_xgb),
    ],
    "pr_auc_preview": [
        average_precision_score(y_val, val_proba_logreg),
        average_precision_score(y_val, val_proba_rf),
        average_precision_score(y_val, val_proba_xgb),
    ],
})
print("Preview cepat (evaluasi LENGKAP ada di notebook 09):")
print(preview)


Preview cepat (evaluasi LENGKAP ada di notebook 09):
                 model  roc_auc_preview  pr_auc_preview
0  Logistic Regression         0.588491        0.044512
1        Random Forest         0.549849        0.021473
2              XGBoost         0.561984        0.016514


---
## 8. Simpan Model & Split Data untuk Notebook 09


In [9]:
with open(MODELS_DIR / "logistic_regression.pkl", "wb") as f:
    pickle.dump(logreg_pipeline, f)
with open(MODELS_DIR / "random_forest.pkl", "wb") as f:
    pickle.dump(rf_pipeline, f)
with open(MODELS_DIR / "xgboost.pkl", "wb") as f:
    pickle.dump(xgb_pipeline, f)

# Simpan juga split data (supaya notebook 09 bisa langsung load, tidak perlu split ulang -
# penting supaya test set benar-benar konsisten sama, tidak ada risiko re-split yang beda)
split_data = {
    "X_train": X_train, "y_train": y_train,
    "X_val": X_val, "y_val": y_val,
    "X_test": X_test, "y_test": y_test,
}
with open(DATA_DIR / "train_val_test_split.pkl", "wb") as f:
    pickle.dump(split_data, f)

print("3 model tersimpan di models/")
print("Split data tersimpan di data_processed/train_val_test_split.pkl")


3 model tersimpan di models/
Split data tersimpan di data_processed/train_val_test_split.pkl


---
## Definition of Done (Fase 7.5-7.7 — Modeling)

- [ ] Missing value ditangani lewat imputasi eksplisit (bukan `fillna(0)` sembarangan)
- [ ] Train/Validation/Test split 3 arah, test set BENAR-BENAR untouched
- [ ] Logistic Regression fitted dengan `class_weight="balanced"`
- [ ] Random Forest fitted dengan `class_weight="balanced"`
- [ ] XGBoost fitted dengan `scale_pos_weight` sesuai imbalance ratio training set
- [ ] SMOTE TIDAK dipakai secara default (sesuai keputusan Decision Gate)
- [ ] 3 model & split data tersimpan untuk evaluasi mendalam

**Lanjut ke:** `09_model_evaluation.ipynb` — di sini baru dilakukan evaluasi PR-AUC lengkap, precision/recall/F1, confusion matrix, dan probability calibration check (Fase 8.1 & 8.2).
